## Visualization dataset

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# RUN_NAME = "dinov2_vitb_distill_196_ipc1_augs6_seathru_v100"
# RUN_NAME = "dinov2_vitb_distill_196_ipc1_augs10_physics_v100"
RUN_NAME = "dinov2_vitb_distill_252_ipc1_augs10_physics_v100_32g"
PROJECT = "/lustre/fswork/projects/rech/rbw/ucw75ke/projects/GradientDistillation"
DATA_PTH = f"{PROJECT}/logged_files/distillation/aqua20/dinov2_vitb/{RUN_NAME}/data.pth"

# map_location='cpu' obligatoire sur le frontal (pas de GPU)
data = torch.load(DATA_PTH, weights_only=False, map_location='cpu')
print("Keys:", list(data.keys()))
print("Images shape:", data["images"].shape)

# Affichage
grid = vutils.make_grid(data["images"], nrow=4, padding=2)
plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Distilled Images")
plt.show()

## Visualization Physics

In [ ]:
import torch
import matplotlib.pyplot as plt

run_name = "dinov2_vitb_distill_252_ipc1_augs10_physics_v100_32g"  # adapte au run réel
data = torch.load(
    f"{PROJECT}/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth",
    weights_only=False,
    map_location='cpu',
)
print(f"Keys in data: {data.keys()}")

img_to_show = 14
J_1 = data["syn_J"][img_to_show]    # (3, H, W)  clean radiance
T_1 = data["syn_T"][img_to_show]    # (1, H, W)  transmission
B_1 = data["syn_B"][img_to_show]    # (3, 1, 1)  background light
I_1 = data["images"][img_to_show]   # (3, H, W)  observed

print("shapes:", J_1.shape, T_1.shape, B_1.shape, I_1.shape)
print("ranges:")
for name, t in [("I", I_1), ("J", J_1), ("T", T_1), ("B", B_1)]:
    print(f"  {name}: min={t.min().item():.3f}  max={t.max().item():.3f}  mean={t.mean().item():.3f}")


def img_to_np(t, normalize=True):
    """(C, H, W) -> (H, W, C) numpy for imshow. Preserves colour by joint min-max."""
    t = t.detach().cpu().float()
    if normalize:
        t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    if t.shape[0] == 1:                # grayscale
        return t.squeeze(0).numpy()
    return t.permute(1, 2, 0).numpy()  # (H, W, 3)


def b_to_swatch(B, size=128, normalize=True):
    """(3, 1, 1) or (3,) -> (size, size, 3) numpy swatch."""
    B = B.detach().cpu().float().squeeze()   # -> (3,)
    swatch = B[:, None, None].expand(3, size, size).contiguous()
    if normalize:
        swatch = (swatch - swatch.min()) / (swatch.max() - swatch.min() + 1e-8)
    return swatch.permute(1, 2, 0).numpy()


fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(img_to_np(I_1))
axes[0, 0].set_title(f"I — observed  {tuple(I_1.shape)}")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_to_np(J_1))
axes[0, 1].set_title(f"J — clean radiance  {tuple(J_1.shape)}")
axes[0, 1].axis("off")

axes[1, 0].imshow(img_to_np(T_1), cmap="gray")
axes[1, 0].set_title(f"T — transmission  {tuple(T_1.shape)}")
axes[1, 0].axis("off")

b_vals = torch.sigmoid(B_1).detach().cpu().float().squeeze().tolist()
axes[1, 1].imshow(b_to_swatch(B_1))
axes[1, 1].set_title(f"B — background  rgb=[{b_vals[0]:.2f}, {b_vals[1]:.2f}, {b_vals[2]:.2f}]")
axes[1, 1].axis("off")

plt.tight_layout()
plt.savefig("physics.png", dpi=150, bbox_inches="tight")
plt.show()

## Visualization sea thru

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

run_name = "dinov2_vitb_distill_252_ipc1_augs10_seathru_v100_32g"  # adapte au run réel
data = torch.load(
    f"{PROJECT}/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth",
    weights_only=False,
    map_location='cpu',
)
print(f"Keys in data: {data.keys()}")

img_to_show = 14
J_1 = data["syn_J"][img_to_show]      # (3, H, W)  clean radiance, in [0, 1]
T_1 = data["syn_T"][img_to_show]      # (3, H, W)  transmission per channel, in (0, 1]
d_1 = data["syn_d"][img_to_show]      # (1, H, W)  depth map, in [0, +inf)
beta_1 = data["syn_beta"][img_to_show]  # (3, 1, 1)  per-channel attenuation, in [0, +inf)
B_1 = data["syn_B"][img_to_show]      # (3, 1, 1)  background light, in [0, 1]
I_1 = data["images"][img_to_show]     # (3, H, W)  observed

print("shapes:", I_1.shape, J_1.shape, T_1.shape, d_1.shape, beta_1.shape, B_1.shape)
print("ranges:")
for name, t in [("I", I_1), ("J", J_1), ("T", T_1), ("d", d_1), ("beta", beta_1), ("B", B_1)]:
    print(f"  {name}: min={t.min().item():.3f}  max={t.max().item():.3f}  mean={t.mean().item():.3f}")


def img_to_np(t, normalize=True):
    """(C, H, W) -> (H, W, C) numpy for imshow."""
    t = t.detach().cpu().float()
    if normalize:
        t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    if t.shape[0] == 1:
        return t.squeeze(0).numpy()
    return t.permute(1, 2, 0).numpy()


def b_to_swatch(B, size=128):
    """(3, 1, 1) or (3,) in [0, 1] -> (size, size, 3) numpy swatch.
    B is already in [0, 1] (decode_B applied sigmoid), no renormalization."""
    B = B.detach().cpu().float().squeeze()  # (3,)
    swatch = B[:, None, None].expand(3, size, size).contiguous()
    return swatch.permute(1, 2, 0).numpy()


fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: I, J, T (composite via mean over channels for visualization)
axes[0, 0].imshow(img_to_np(I_1))
axes[0, 0].set_title(f"I — observed  {tuple(I_1.shape)}")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_to_np(J_1))
axes[0, 1].set_title(f"J — clean radiance  {tuple(J_1.shape)}")
axes[0, 1].axis("off")

# T is now 3-channel; show the mean across channels as grayscale
T_mean = T_1.mean(dim=0, keepdim=True)
axes[0, 2].imshow(img_to_np(T_mean), cmap="gray", vmin=0, vmax=1)
axes[0, 2].set_title(f"T — transmission (mean over RGB)  {tuple(T_1.shape)}")
axes[0, 2].axis("off")

# Row 2: depth map d, B swatch, beta barplot
d_np = d_1.squeeze(0).detach().cpu().float().numpy()
im = axes[1, 0].imshow(d_np, cmap="viridis")
axes[1, 0].set_title(f"d — depth  {tuple(d_1.shape)}\n(min={d_np.min():.2f}, max={d_np.max():.2f})")
axes[1, 0].axis("off")
plt.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

b_vals = B_1.detach().cpu().float().squeeze().tolist()
axes[1, 1].imshow(b_to_swatch(B_1))
axes[1, 1].set_title(f"B — background\nrgb=[{b_vals[0]:.2f}, {b_vals[1]:.2f}, {b_vals[2]:.2f}]")
axes[1, 1].axis("off")

beta_vals = beta_1.detach().cpu().float().squeeze().numpy()  # (3,)
colors = ["#d62728", "#2ca02c", "#1f77b4"]  # R, G, B
bars = axes[1, 2].bar(["R", "G", "B"], beta_vals, color=colors)
axes[1, 2].set_title(
    f"β — attenuation per channel\n"
    f"R={beta_vals[0]:.3f}, G={beta_vals[1]:.3f}, B={beta_vals[2]:.3f}"
)
axes[1, 2].set_ylabel("β")
axes[1, 2].grid(axis="y", alpha=0.3)
# annotate
for bar, val in zip(bars, beta_vals):
    axes[1, 2].text(
        bar.get_x() + bar.get_width() / 2,
        val,
        f"{val:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig("seaThru.png", dpi=150, bbox_inches="tight")
plt.show()